# Maternal Engagement EF — On-nest vs Off-nest (3-band frequency)

Paper-active dCSFA-NMF model for separating on-nest vs off-nest LFP windows in 3-band frequency resolution. Eight sections, each one or two function calls against `src/` plus a one-line printed summary.

* Final model: `Maternal_model_TrainC_Onnest_Mar27_ver2.pt`
* Hyperparameters from LOO validation (sup_weight=0.05, n_epochs=400, batch_size=256, seed=42)
* Stage backproject artifact: `OnnestEF_3band.xlsx`

In [ ]:
# Allow imports from ../src
import sys, os, pickle
import numpy as np
import torch
import matplotlib.pyplot as plt

_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if os.path.join(_repo_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_repo_root, "src"))

from electome.data_utils import create_period_dataset
from electome.training import run_loo_cv, train_final_model
from electome.analysis import process_W_nmf_dual_filter
from electome.viz import create_bar_heatmap_selective
from electome.workflow import validate_on_ELS, run_circos_prep, run_stage_backproject
from electome.sara_requests import (sara_pup_retrieval, sara_onnest_loading_inspect,
                            sara_p3_behavior)


## Section 1. Data loading and processing

In [ ]:
# Configuration -----------------------------------------------------------
TRAINING_DATA_FILE = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combined/full_onnest_spec_features_8roi_Jan212026_Trim.pkl"

# Raw mouse ids as they appear in the pkl (note the underscore between
# the cohort letter and the ELS number).
C_MOUSE_IDS_RAW = ["C7_ELS11", "C2_ELS18", "C5_ELS20", "C7_ELS22",
                    "C1_ELS32", "C5_ELS40", "C6_ELS42", "C7_ELS45"]
E_MOUSE_IDS_RAW = ["E1_ELS33", "E2_ELS3", "E3_ELS37", "E4_ELS39",
                    "E5_ELS41", "E6_ELS44"]

# Canonical (underscore-stripped) ids -- used by the stage-backproject section
C_MOUSE_IDS = [m.replace("_", "") for m in C_MOUSE_IDS_RAW]
E_MOUSE_IDS = [m.replace("_", "") for m in E_MOUSE_IDS_RAW]

PERIODS_TO_KEEP = ["P1", "P3", "P8"]

# Load + filter -----------------------------------------------------------
with open(TRAINING_DATA_FILE, "rb") as f:
    train_dict = pickle.load(f)

base_mask = np.isin(train_dict["period"], PERIODS_TO_KEEP)
base_data = {k: v[base_mask] for k, v in train_dict.items()
             if isinstance(v, np.ndarray) and len(v) == len(train_dict["period"])}
base_y = base_data["onnest_label"]

# Build datasets (Train C uses P1+P3; tests use P8) -----------------------
train_c     = create_period_dataset(base_data, base_y, C_MOUSE_IDS_RAW, ["P1", "P3"], "Train C",     verbose=False)
test_c_P8   = create_period_dataset(base_data, base_y, C_MOUSE_IDS_RAW, ["P8"],       "Test C P8",   verbose=False)
test_e_P1P3 = create_period_dataset(base_data, base_y, E_MOUSE_IDS_RAW, ["P1", "P3"], "Test E P1P3", verbose=False)
test_e_P8   = create_period_dataset(base_data, base_y, E_MOUSE_IDS_RAW, ["P8"],       "Test E P8",   verbose=False)

print(f"Train C     : X={train_c['X'].shape},     mice={list(train_c['mouse_list'])}")
print(f"Test C P8   : X={test_c_P8['X'].shape},   mice={list(test_c_P8['mouse_list'])}")
print(f"Test E P1P3 : X={test_e_P1P3['X'].shape}, mice={list(test_e_P1P3['mouse_list'])}")
print(f"Test E P8   : X={test_e_P8['X'].shape},   mice={list(test_e_P8['mouse_list'])}")


## Section 2. LOO training

Pure leave-one-mouse-out cross-validation (no validation split). Reports per-mouse AUC and a one-sided Wilcoxon signed-rank test against chance (`H1: AUC > 0.5`). No plots.

In [ ]:
# Hyperparameters (from LOO validation; same dict reused in Section 3) ---
SEED = 42
N_EPOCHS = 400
N_PRE_EPOCHS = 100
NMF_MAX_ITER = 100
BATCH_SIZE = 256
LR = 1e-3

MODEL_PARAMS = {
    "n_components": 10,
    "n_sup_networks": 1,
    "optim_name": "SGD",
    "recon_loss": "MSE",
    "sup_recon_weight": 0.0,
    "sup_weight": 0.05,
    "phi_weight": 0,
    "n_intercepts": 1,
    "useDeepEnc": True,
    "h": 64,
    "sup_recon_type": "Residual",
    "feature_groups": None,
    "group_weights": None,
    "fixed_corr": "Positive",
    "momentum": 0.9,
    "sup_smoothness_weight": 1,
}

# Run LOO ----------------------------------------------------------------
loo = run_loo_cv(
    train_c["X"], train_c["y"], train_c["y_intercept"],
    model_params=MODEL_PARAMS,
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, lr=LR,
    n_pre_epochs=N_PRE_EPOCHS, nmf_max_iter=NMF_MAX_ITER,
    seed=SEED, n_jobs=4,
)
print(loo.summary())
print()
print(loo.per_mouse_table())


## Section 3. Full training (paper model)

Final model trained on all C mice (P1+P3) with the LOO-validated hyperparameters. Saved as the paper-active model file.

In [ ]:
MODEL_SAVE_FILE = "Maternal_model_TrainC_Onnest_Mar27_ver2.pt"
MODEL_STATE_DICT = "Maternal_sd_TrainC_Onnest_Mar27_ver2.pt"

model = train_final_model(
    train_c["X"], train_c["y"], train_c["y_sampling"],
    model_params=MODEL_PARAMS,
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, lr=LR,
    n_pre_epochs=N_PRE_EPOCHS, nmf_max_iter=NMF_MAX_ITER,
    seed=SEED,
    save_to=MODEL_SAVE_FILE,
    state_dict_to=MODEL_STATE_DICT,
)

train_aucs = [auc[0] for auc in model.train_auc_hist]
print(f"Paper model saved : {MODEL_SAVE_FILE}")
print(f"  Final train AUC : {train_aucs[-1]:.4f}")
print(f"  Final phi (Softplus)   : {model.get_phi(0).item():.4f}")
print(f"  Final beta (intercept) : {model.beta_list[0].item():.4f}")


## Section 4. Circos plot

Selected-feature matrix (top entries by cumulative squared-L2 contribution at `threshold_ratio=0.8`) written to CSV for the external circos plotter.

In [ ]:
df_circos = run_circos_prep(
    model, train_dict,
    output_csv="OnnestEF_3band_circos_input.csv",
    k=0, threshold_ratio=0.8,
)


## Section 5. Elements selection

Dual-filter feature selection on factor 0:

* Absolute strength — cumulative squared-L2 contribution ≥ 0.9 of total.
* Relative uniqueness — factor 0 contributes > 50% of the column sum across factors.

Bar heatmap highlights the intersection in Iowa Gold.

In [ ]:
abs_cut, rel_cut, both_cut, abs_full, rel_full = process_W_nmf_dual_filter(
    model.get_W_nmf(), train_dict,
    abs_cum_ratio=0.9, rel_val=0.5,
    verbose=False,
)

fig = create_bar_heatmap_selective(abs_full, abs_cut, rel_full, rel_cut, both_cut)
fig.savefig("OnnestEF_3band_bar_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Bar heatmap saved : OnnestEF_3band_bar_heatmap.png")
print(f"  Absolute filter        : {(~abs_cut.isna()).sum().sum()} features")
print(f"  Relative-uniqueness    : {(~rel_cut.isna()).sum().sum()} features")
print(f"  Intersection (Gold)    : {(~both_cut.isna()).sum().sum()} features")


## Section 6. Validation on ELS group

Apply the C-trained paper model to held-out C P8 windows and to all E (ELS) windows. Reports per-mouse AUC mean ± SEM and one-sided Wilcoxon test against chance, per dataset. Nothing else.

In [ ]:
els = validate_on_ELS(model, {
    "C mice P8 (held-out)": test_c_P8,
    "E mice P1+P3":         test_e_P1P3,
    "E mice P8":            test_e_P8,
})
print(els.summary())


## Section 7. Stage Backprojection Scores

Project the trained model onto every available developmental stage (Pre, P1, P3, P4, P8, P14, P20) and emit the paper-figure median+IQR plot plus a 10-sheet workbook (`OnnestEF_3band.xlsx`) with raw windows, per-mouse-per-stage detail, mean+SEM, and median+IQR tables.

P4 is restricted to the first 20 minutes (400 windows) per mouse, matching the convention used throughout the paper.

In [ ]:
BACKPROJECT_DATA_FILE = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combine/full_spec_features_8roi.pkl"

backproject = run_stage_backproject(
    model,
    backproject_data_file=BACKPROJECT_DATA_FILE,
    c_mouse_ids=C_MOUSE_IDS,
    e_mouse_ids=E_MOUSE_IDS,
    output_xlsx="OnnestEF_3band.xlsx",
)
print(backproject.summary())


## Section 8. Additional backprojections (Sara's request)

Off-paper one-off data exports requested by Sara. Each call writes one workbook and prints a one-line success message.

* `Onnest_pups.xlsx` — per-mouse loading score at P4 home, split by baseline (first 20 min) / trial-no-retrieval / retrieval / other.
* `OnNestEF_OnOff.xlsx` — per-(period, mouse, onnest_label) loading-score statistics across all stages.
* `OnNestEF_Behavior_P3.xlsx` — P3 loading-score summary restricted to `licking == 1` and `nursing == 1` samples.

In [ ]:
PUP_RETRIEVAL_DATA_FILE = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P4_pup_retrieval_detail.pkl"

# C6ELS9 only appears in the pup-retrieval pkl (not in the training pkl) --
# include it here so the per-mouse Excel includes its row.
sara_pup_retrieval(
    model,
    pup_retrieval_data_file=PUP_RETRIEVAL_DATA_FILE,
    c_mouse_ids=["C6ELS9"] + C_MOUSE_IDS,
    e_mouse_ids=E_MOUSE_IDS,
    output_xlsx="Onnest_pups.xlsx",
)


In [ ]:
sara_onnest_loading_inspect(
    model,
    training_data_file=TRAINING_DATA_FILE,
    output_xlsx="OnNestEF_OnOff.xlsx",
)


In [ ]:
P3_BEHAVIOR_DATA_FILE = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P3_onnest_lick_selfgroom_nurse/full_all_behaviors_complete_trimmed.pkl"

sara_p3_behavior(
    model,
    training_data_file=P3_BEHAVIOR_DATA_FILE,
    output_xlsx="OnNestEF_Behavior_P3.xlsx",
)
